# SitePulse — Construction Anomaly Detection
### Phase 4: Hybrid Unsupervised Anomaly Detection Engine

> **Simulation Disclaimer:**
> This notebook evaluates anomaly detection algorithms on synthetic construction fitout simulation datasets. The evaluation serves to establish algorithmic benchmarks and pipeline verification and does not represent real-world customer field deployments.

## 1. Setup & Ingestion of Historical Daily Trajectories

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

DATA_DIR = "../data/synthetic"
MODEL_DIR = "../model/anomaly"

trajectories_df = pd.read_csv(os.path.join(DATA_DIR, "consolidated_trajectories.csv"))
with open(os.path.join(DATA_DIR, "split_metadata.json")) as f: 
    split_meta = json.load(f)

train_pids = set(split_meta["train_projects"])
val_pids = set(split_meta["val_projects"])
test_pids = set(split_meta["test_projects"])

train_df = trajectories_df[trajectories_df["project_id"].isin(train_pids)]
val_df = trajectories_df[trajectories_df["project_id"].isin(val_pids)]
test_df = trajectories_df[trajectories_df["project_id"].isin(test_pids)]

print(f"Loaded Trajectories: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")
print(f"Ground-truth Anomalies: Val={val_df['is_anomaly'].sum()}, Test={test_df['is_anomaly'].sum()}")

## 2. Load Trained Hybrid Isolation Forest Artifacts

The unsupervised model was trained strictly on normal observations (`is_anomaly == 0`), ensuring no label exposure.

In [ ]:
model_artifact = joblib.load(os.path.join(MODEL_DIR, "isolation_forest.joblib"))
iso_forest = model_artifact["iso_forest"]
scaler = model_artifact["scaler"]
feature_names = model_artifact["features"]

with open(os.path.join(MODEL_DIR, "anomaly_config.json")) as f:
    anomaly_cfg = json.load(f)

print("Trained Anomaly Features:", feature_names)
print("Optimal Threshold Selected on Val:", anomaly_cfg["optimal_threshold"])

## 3. Evaluation on Untouched Test Partition

We score the test partition using the hybrid engine and evaluate detection metrics against ground-truth labels.

In [ ]:
def prepare_features(df):
    d = df.copy()
    d["productivity_ratio"] = d["daily_delta_pct"] / np.maximum(1, d["workers_present"])
    d["shortfall_workers"] = np.maximum(0, d["workers_required"] - d["workers_present"])
    d["has_active_hold"] = d["reason"].isin(["Material Delay", "Client Hold", "Waiting on Another Trade"]).astype(float)
    d["unexplained_zero_progress"] = ((d["has_active_hold"] == 0) & (d["workers_present"] >= 4) & (d["daily_delta_pct"] <= 0.05)).astype(float)
    return d[feature_names]

X_test = prepare_features(test_df)
X_test_scaled = scaler.transform(X_test)
raw_decisions = iso_forest.decision_function(X_test_scaled)
iso_scores = 1.0 / (1.0 + np.exp(raw_decisions * 45.0))

test_scores = []
for idx, (_, r) in enumerate(test_df.iterrows()):
    delta = r["daily_delta_pct"]
    w = r["workers_present"]
    st = r["consecutive_stagnant"]
    hold = r["reason"] in ["Material Delay", "Client Hold", "Waiting on Another Trade"]
    r_score = 0.0
    if delta >= 14.0:
        r_score = 0.98
    elif not hold and w >= 4 and delta <= 0.05 and st >= 1:
        r_score = 0.95
    elif w == 0 and delta >= 2.0:
        r_score = 0.96
    test_scores.append(max(iso_scores[idx], r_score))

test_scores = np.array(test_scores)
y_test = test_df["is_anomaly"].values
test_preds = (test_scores >= anomaly_cfg["optimal_threshold"]).astype(int)

print("=== TEST SET ANOMALY DETECTION REPORT ===")
print(f"Precision: {precision_score(y_test, test_preds) * 100:.2f}%")
print(f"Recall   : {recall_score(y_test, test_preds) * 100:.2f}%")
print(f"F1-Score : {f1_score(y_test, test_preds):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_preds))

## 4. Inspection of Flagged High-Severity Anomalies

In [ ]:
test_df_results = test_df.copy()
test_df_results["anomaly_score"] = test_scores
test_df_results["predicted_anomaly"] = test_preds

flagged = test_df_results[test_df_results["predicted_anomaly"] == 1][[
    "project_id", "trade_name", "day_number", "daily_delta_pct",
    "workers_present", "workers_required", "anomaly_score", "anomaly_type"
]].sort_values("anomaly_score", ascending=False)

print(f"Total Anomalies Flagged in Test Set: {len(flagged)}")
display(flagged.head(15))